# Imports 

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    LongType, DoubleType, TimestampType, DateType, BooleanType
)
from pyspark.sql.window import Window

spark = SparkSession.builder.getOrCreate()

# Config

In [0]:
CATALOG = "devs"
SCHEMA_BRONZE = "bronze_william_lyagoba1"
SCHEMA_SILVER = "silver_william_lyagoba1"
BRONZE_TABLE = f"{CATALOG}.{SCHEMA_BRONZE}.audit_episodes"
SILVER_TABLE = f"{CATALOG}.{SCHEMA_BRONZE}.audit_episodes"
ERROR_TABLE = f"{CATALOG}.{SILVER_TABLE}.audit_episodes_cast_errors"
PARTITION_BY = "bso_organisation_code"   # partition Silver by BSO org
DATE_FMT = "yyyy-MM-dd"
TS_FMT   = "yyyy-MM-dd HH:mm:ss"
DEDUPE_KEYS  = ["episode_id"] # Columns that define a unique record for deduplication

# Read Bronze

In [0]:
df_bronze = spark.read.format("delta").table(BRONZE_TABLE)
df_bronze.printSchema()        # Remove later
df_bronze.dtypes               # Remove later

root
 |-- nhs_number: long (nullable = true)
 |-- episode_id: integer (nullable = true)
 |-- appointment_made: boolean (nullable = true)
 |-- date_of_foa: date (nullable = true)
 |-- date_of_as: date (nullable = true)
 |-- episode_date: date (nullable = true)
 |-- early_recall_date: date (nullable = true)
 |-- change_db_date_time: timestamp (nullable = true)
 |-- episode_type: string (nullable = true)
 |-- end_code: string (nullable = true)
 |-- call_recall_status_authorised_by: string (nullable = true)
 |-- end_code_last_updated: timestamp (nullable = true)
 |-- bso_organisation_code: string (nullable = true)
 |-- bso_batch_id: string (nullable = true)
 |-- reason_closed_code: string (nullable = true)
 |-- end_point: string (nullable = true)
 |-- final_action_code: string (nullable = true)
 |-- _source_file_name: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)



[('nhs_number', 'bigint'),
 ('episode_id', 'int'),
 ('appointment_made', 'boolean'),
 ('date_of_foa', 'date'),
 ('date_of_as', 'date'),
 ('episode_date', 'date'),
 ('early_recall_date', 'date'),
 ('change_db_date_time', 'timestamp'),
 ('episode_type', 'string'),
 ('end_code', 'string'),
 ('call_recall_status_authorised_by', 'string'),
 ('end_code_last_updated', 'timestamp'),
 ('bso_organisation_code', 'string'),
 ('bso_batch_id', 'string'),
 ('reason_closed_code', 'string'),
 ('end_point', 'string'),
 ('final_action_code', 'string'),
 ('_source_file_name', 'string'),
 ('_ingestion_timestamp', 'timestamp')]

# Type Casting - Subjects

In [0]:
df_cast = (
    df_bronze

    # numeric 
    .withColumn("nhs_number", F.col("nhs_number").cast(LongType()))
    .withColumn("episode_id", F.col("episode_id").cast(LongType()))

    # booleans 
    .withColumn("appointment_made",
        F.col("appointment_made").cast(BooleanType()))

    # dates
    .withColumn("date_of_foa", F.to_date(F.col("date_of_foa"), DATE_FMT))
    .withColumn("date_of_as", F.to_date(F.col("date_of_as"), DATE_FMT))
    .withColumn("episode_date", F.to_date(F.col("episode_date"), DATE_FMT))
    .withColumn("early_recall_date", F.to_date(F.col("early_recall_date"), DATE_FMT))

    # timestamps
    .withColumn("change_db_date_time", F.to_timestamp(F.col("change_db_date_time"), TS_FMT))
    .withColumn("end_code_last_updated", F.to_timestamp(F.col("end_code_last_updated"), TS_FMT))

    # strings
    .withColumn("episode_type", F.col("episode_type").cast("string"))
    .withColumn("end_code", F.col("end_code").cast("string"))
    .withColumn("call_recall_status_authorised_by", F.col("call_recall_status_authorised_by").cast("string"))
    .withColumn("bso_organisation_code", F.col("bso_organisation_code").cast("string"))
    .withColumn("bso_batch_id", F.col("bso_batch_id").cast("string"))
    .withColumn("reason_closed_code", F.col("reason_closed_code").cast("string"))
    .withColumn("end_point", F.col("end_point").cast("string"))
    .withColumn("final_action_code", F.col("final_action_code").cast("string"))
)

# Quarantine rows with critical cast failures

In [0]:
CRITICAL_NOT_NULL = [
    "nhs_number",
    "episode_id",
    "episode_date",
    "change_db_date_time",
    "end_code_last_updated",
]

invalid_filter = F.lit(False)
for c in CRITICAL_NOT_NULL:
    invalid_filter = invalid_filter | F.col(c).isNull()

df_invalid = df_cast.filter(invalid_filter)
invalid_count = df_invalid.count()

if invalid_count > 0:
    print(f"{invalid_count:,} rows failed critical casting, quarantined to {ERROR_TABLE}")
    (
        df_invalid.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(ERROR_TABLE)
    )
else:
    print("No critical cast failures found.")

df_valid = df_cast.filter(~invalid_filter)

No critical cast failures found.


# Deduplication

In [0]:
window = (
    Window
    .partitionBy(*DEDUPE_KEYS)
    .orderBy(F.col("_ingestion_timestamp").desc())
)

df_silver = (
    df_valid
    .withColumn("_row_num", F.row_number().over(window))
    .filter(F.col("_row_num") == 1)
    .drop("_row_num")
)

dupe_count = df_valid.count() - df_silver.count()
print(f"Duplicates removed: {dupe_count:,}")

Duplicates removed: 0


# Write to Sliver table

In [0]:
(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy(PARTITION_BY)
    .saveAsTable(SILVER_TABLE)
)

print(f"Silver load complete.")

Silver load complete.
